# Task 4D: GAN-Based Data Synthesis and Robustness Audit for Kannada Numeral Recognition

**MGIT (Autonomous), Hyderabad** | Deep Learning Complex Engineering Problem | **Variant 4D (Kannada MNIST)**  
**Team 16** | CM701PC / CM751PC | Academic Year 2026–27

**Status of this notebook:** code is complete and runnable. Every number in your report must come from *your* execution of this notebook.

**Two research questions**
1. Does adding GAN-generated numerals improve recognition, especially on the three weakest classes?
2. How vulnerable is the CNN to FGSM, and does adversarial training restore robustness without hurting clean accuracy by more than 2 percentage points?

**How to run**
1. Runtime → Change runtime type → **GPU**.
2. First run with `QUICK_CHECK = True` (a few minutes) to confirm nothing crashes.
3. Then set `QUICK_CHECK = False` and run all cells for submission results (see README for time estimates).

**Sections:** 1 Setup → 2 Data → 3 Inspection → 4 Baseline CNN → 5 cGAN → 6 Synthetic quality → 7 Mixing ratios → 8 FGSM → 9 Adversarial training → 10 Tables → 11 Target check → 12 Export


## 1. Environment setup, configuration, seeds

In [ ]:
# If you uploaded the project ZIP, unzip it first (Colab)
import os, zipfile, glob
if os.path.exists('/content'):
    zips = glob.glob('/content/*.zip') + glob.glob('/content/DL_T16*.zip')
    for z in zips:
        if 'Kannada' in z or 'T16' in z or 'DL_T16' in z:
            print('Extracting', z)
            with zipfile.ZipFile(z) as zf:
                zf.extractall('/content')
            break
    # Make project root importable
    for candidate in ['/content/DL_T16_4D_Kannada_MNIST', '/content']:
        if os.path.isdir(os.path.join(candidate, 'src')):
            os.chdir(candidate)
            break
print('CWD:', os.getcwd())


In [ ]:
import os, sys, time, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Install only if missing
try:
    import tensorflow as tf
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow'])
    import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# CONFIGURATION – change only this cell for quick vs full runs
# ---------------------------------------------------------------------------
QUICK_CHECK = True          # True = tiny run to test the code. MUST be False for submission.
SEED = 42

if QUICK_CHECK:
    CFG = dict(
        n_real_per_class=200, clf_epochs=4, gan_epochs=6,
        adv_epochs=3, batch=64, gan_batch=64, latent=100, patience=3,
    )
else:
    CFG = dict(
        n_real_per_class=600, clf_epochs=25, gan_epochs=50,
        adv_epochs=15, batch=128, gan_batch=128, latent=100, patience=6,
    )

EPS_LIST = [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]
MAIN_EPS = 0.10
SYNTHETIC_TOTAL = 25000 if not QUICK_CHECK else 2000
SYNTHETIC_RATIOS = [0.0, 0.10, 0.25]

OUT_DIR = 'outputs'
for sub in ['figures', 'metrics', 'tables', 'generated_samples', 'models']:
    os.makedirs(f'{OUT_DIR}/{sub}', exist_ok=True)
TAG = 'QUICKCHECK_NOT_FOR_SUBMISSION' if QUICK_CHECK else 'FINAL'

# Seeds
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
print('Mode:', TAG, '| CFG:', CFG)
if QUICK_CHECK:
    print('*** QUICK_CHECK is ON – do NOT report these numbers ***')


## 2. Dataset acquisition and verification

Source: Prabhu, *Kannada-MNIST* (arXiv:1908.01242). Zenodo record 3359691.  
We use the original 60 000 / 10 000 split and **ignore Dig-MNIST**.


In [ ]:
# Prefer the modular loader if the project package is present
sys.path.insert(0, os.getcwd())
try:
    from src.dataset import load_kannada_mnist
    raw = load_kannada_mnist()
    X_tr_raw, y_tr_raw = raw['X_train'], raw['y_train']
    X_te_raw, y_te_raw = raw['X_test'], raw['y_test']
except Exception as e:
    print('Modular loader failed, using inline fallback:', e)
    import urllib.request, zipfile, glob as _glob
    DATA_DIR = 'data'
    os.makedirs(DATA_DIR, exist_ok=True)
    ZIP_URL = 'https://zenodo.org/records/3359691/files/Kannada_MNIST_npz.zip'
    zpath = f'{DATA_DIR}/Kannada_MNIST_npz.zip'
    if not _glob.glob(f'{DATA_DIR}/**/*.npz', recursive=True):
        if not os.path.exists(zpath):
            print('Downloading…')
            req = urllib.request.Request(ZIP_URL, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=180) as r, open(zpath, 'wb') as f:
                f.write(r.read())
        with zipfile.ZipFile(zpath) as z:
            z.extractall(DATA_DIR)
    files = sorted(_glob.glob(f'{DATA_DIR}/**/*.npz', recursive=True))
    print('Files:', [os.path.basename(f) for f in files])
    def pick(kind, split):
        for f in files:
            b = os.path.basename(f).lower()
            if kind in b and split in b and 'dig' not in b:
                return f
        raise FileNotFoundError(kind + '/' + split)
    def load_np(f):
        d = np.load(f); a = d[d.files[0]]
        return a.reshape(-1, 28, 28) if a.ndim == 2 and a.shape[1] == 784 else a
    X_tr_raw = load_np(pick('x', 'train'))
    y_tr_raw = load_np(pick('y', 'train')).astype(np.int32).ravel()
    X_te_raw = load_np(pick('x', 'test'))
    y_te_raw = load_np(pick('y', 'test')).astype(np.int32).ravel()

for name, X, y in [('train', X_tr_raw, y_tr_raw), ('test', X_te_raw, y_te_raw)]:
    assert X.shape[1:] == (28, 28), name
    assert set(np.unique(y)) == set(range(10)), name
    print(f'{name:6s} X{X.shape}  dtype={X.dtype}  min={X.min()} max={X.max()}  '
          f'counts={np.bincount(y.astype(int)).tolist()}')
assert len(X_tr_raw) == 60000 and len(X_te_raw) == 10000, 'Unexpected sizes – investigate'


## 3. Preprocessing, splits and inspection

* Pixels → [0, 1], shape (N, 28, 28, 1).  
* Validation = stratified 10 % of official train.  
* Test = official 10 000 images, never used for selection.  
* Scarce-data simulation: class-balanced subset of `n_real_per_class` images.


In [ ]:
def prep(X):
    return (X.astype('float32') / 255.0)[..., None]

X_tr_all = prep(X_tr_raw)
y_tr_all = y_tr_raw.astype(np.int32)
X_te = prep(X_te_raw)
y_te = y_te_raw.astype(np.int32)
TEST_N = len(X_te)

X_pool, X_val, y_pool, y_val = train_test_split(
    X_tr_all, y_tr_all, test_size=0.10, stratify=y_tr_all, random_state=SEED)

def subsample_per_class(X, y, n, seed):
    if n is None:
        return X, y
    rng = np.random.default_rng(seed)
    idx = np.concatenate([
        rng.choice(np.where(y == c)[0], size=min(n, (y == c).sum()), replace=False)
        for c in range(10)
    ])
    rng.shuffle(idx)
    return X[idx], y[idx]

X_real, y_real = subsample_per_class(X_pool, y_pool, CFG['n_real_per_class'], SEED)
N_REAL = len(X_real)
print(f'real train={N_REAL}  val={len(X_val)}  test={len(X_te)}')

def make_mosaic(X, y, per_class=8):
    rows = []
    for c in range(10):
        imgs = X[y == c][:per_class, ..., 0]
        if len(imgs) < per_class:
            imgs = np.concatenate([imgs, np.zeros((per_class - len(imgs), 28, 28))])
        rows.append(np.concatenate(list(imgs), axis=1))
    return np.concatenate(rows, axis=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(make_mosaic(X_real, y_real), cmap='gray')
ax[0].set_title('Real training examples (row = class 0..9)')
ax[0].axis('off')
pd.Series(np.bincount(y_real, minlength=10)).plot.bar(ax=ax[1])
ax[1].set_title('Class counts (real train used)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/01_real_and_counts.png', dpi=120)
plt.show()


## 4. Baseline CNN

Small CNN on purpose. Weakest classes chosen from **validation** F1 only.


In [ ]:
loss_obj = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
loss_per_sample = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def build_cnn(seed=0, name='cnn'):
    keras.utils.set_random_seed(seed)
    return keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(128, activation='relu', name='feat'),
        layers.Dropout(0.4),
        layers.Dense(10, name='logits'),
    ], name=name)

def train_classifier(X, y, seed, epochs=None, name='clf'):
    model = build_cnn(seed, name=name)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss_obj, metrics=['accuracy'])
    t0 = time.time()
    hist = model.fit(
        X, y, validation_data=(X_val, y_val),
        epochs=epochs or CFG['clf_epochs'], batch_size=CFG['batch'],
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=CFG['patience'], restore_best_weights=True)],
        verbose=1)
    return model, hist, time.time() - t0

def evaluate(model, X, y):
    pred = model.predict(X, batch_size=512, verbose=0).argmax(1)
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=list(range(10)), zero_division=0)
    return dict(
        acc=float((pred == y).mean()), macro_f1=float(f.mean()),
        precision=p, recall=r, f1=f, support=s,
        cm=confusion_matrix(y, pred, labels=list(range(10))), pred=pred)

def plot_cm(cm, title, path):
    plt.figure(figsize=(5.5, 5))
    plt.imshow(cm, cmap='Blues')
    plt.title(title); plt.xlabel('predicted'); plt.ylabel('true')
    for i in range(10):
        for j in range(10):
            plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.xticks(range(10)); plt.yticks(range(10))
    plt.tight_layout(); plt.savefig(path, dpi=120); plt.show()

clf_base, hist_base, secs_base = train_classifier(X_real, y_real, seed=0, name='baseline')
ev_val_base = evaluate(clf_base, X_val, y_val)
ev_test_base = evaluate(clf_base, X_te, y_te)
print(f'Baseline: val_acc={ev_val_base["acc"]:.4f}  test_acc={ev_test_base["acc"]:.4f}  '
      f'macro-F1={ev_test_base["macro_f1"]:.4f}  ({secs_base:.0f}s)')

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, k in zip(ax, ['loss', 'accuracy']):
    a.plot(hist_base.history[k], label='train')
    a.plot(hist_base.history['val_' + k], label='val')
    a.set_title('baseline ' + k); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/02_baseline_curves.png', dpi=120); plt.show()
plot_cm(ev_test_base['cm'], 'Baseline CM (test)', f'{OUT_DIR}/figures/03_baseline_cm.png')

WEAK = [int(c) for c in np.argsort(ev_val_base['f1'])[:3]]
print('Three weakest classes (validation F1):', WEAK)
print(pd.DataFrame({
    'precision': ev_test_base['precision'], 'recall': ev_test_base['recall'],
    'f1': ev_test_base['f1']}).round(4))
clf_base.save(f'{OUT_DIR}/models/baseline.keras')


## 5. Conditional GAN (cGAN)

In [ ]:

LATENT = CFG['latent']

def build_generator(latent_dim=LATENT, num_classes=10):
    noise = keras.Input(shape=(latent_dim,), name='noise')
    label = keras.Input(shape=(1,), dtype='int32', name='label')
    emb = layers.Embedding(num_classes, 50)(label)
    emb = layers.Flatten()(emb)
    x = layers.Concatenate()([noise, emb])
    x = layers.Dense(7 * 7 * 128, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((7, 7, 128))(x)
    x = layers.Conv2DTranspose(128, 4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    img = layers.Conv2D(1, 3, padding='same', activation='tanh')(x)
    return keras.Model([noise, label], img, name='generator')

def build_discriminator(num_classes=10):
    img = keras.Input(shape=(28, 28, 1), name='image')
    label = keras.Input(shape=(1,), dtype='int32', name='label')
    emb = layers.Embedding(num_classes, 28 * 28)(label)
    emb = layers.Reshape((28, 28, 1))(emb)
    x = layers.Concatenate()([img, emb])
    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1)(x)
    return keras.Model([img, label], x, name='discriminator')

generator = build_generator()
discriminator = build_discriminator()
g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
bce = keras.losses.BinaryCrossentropy(from_logits=True)

@tf.function
def gan_train_step(real_images, real_labels):
    batch = tf.shape(real_images)[0]
    noise = tf.random.normal((batch, LATENT))
    with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
        fake_images = generator([noise, real_labels], training=True)
        real_logits = discriminator([real_images, real_labels], training=True)
        fake_logits = discriminator([fake_images, real_labels], training=True)
        d_loss = bce(tf.ones_like(real_logits) * 0.9, real_logits) + bce(tf.zeros_like(fake_logits), fake_logits)
        g_loss = bce(tf.ones_like(fake_logits), fake_logits)
    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))
    return d_loss, g_loss

# Fixed noise for consistent grids
fixed_noise = tf.random.normal((100, LATENT), seed=SEED)
fixed_labels = tf.constant(np.repeat(np.arange(10), 10).reshape(-1, 1), dtype=tf.int32)

def save_gan_grid(epoch):
    fake = generator.predict([fixed_noise, fixed_labels], verbose=0)
    fake = np.clip((fake + 1) / 2, 0, 1)
    path = f'{OUT_DIR}/figures/gan_samples_epoch_{epoch:03d}.png'
    plt.figure(figsize=(10, 10))
    plt.imshow(make_mosaic(fake, fixed_labels.numpy().ravel(), per_class=10), cmap='gray')
    plt.title(f'cGAN samples – epoch {epoch}'); plt.axis('off')
    plt.tight_layout(); plt.savefig(path, dpi=120); plt.close()
    print('Saved', path)

# Train
X_gan = X_real * 2.0 - 1.0   # [0,1] → [-1,1]
n = len(X_gan)
bs = CFG['gan_batch']
steps = max(1, n // bs)
gan_log = {'d_loss': [], 'g_loss': []}
print(f'Training cGAN for {CFG["gan_epochs"]} epochs …')
for epoch in range(1, CFG['gan_epochs'] + 1):
    idx = np.random.permutation(n)
    d_losses, g_losses = [], []
    for s in range(steps):
        bi = idx[s*bs:(s+1)*bs]
        imgs = tf.convert_to_tensor(X_gan[bi], tf.float32)
        labs = tf.convert_to_tensor(y_real[bi].reshape(-1, 1), tf.int32)
        dl, gl = gan_train_step(imgs, labs)
        d_losses.append(float(dl)); g_losses.append(float(gl))
    gan_log['d_loss'].append(float(np.mean(d_losses)))
    gan_log['g_loss'].append(float(np.mean(g_losses)))
    print(f'Epoch {epoch:3d}/{CFG["gan_epochs"]}  D={gan_log["d_loss"][-1]:.4f}  G={gan_log["g_loss"][-1]:.4f}')
    if epoch in (1, max(1, CFG['gan_epochs']//2), CFG['gan_epochs']):
        save_gan_grid(epoch)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gan_log['d_loss'], label='D loss'); ax.plot(gan_log['g_loss'], label='G loss')
ax.legend(); ax.set_title('cGAN losses'); ax.set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/04_gan_losses.png', dpi=120); plt.show()
generator.save(f'{OUT_DIR}/models/cgan_generator.keras')


## 6. Generate synthetic images and quality audit

In [ ]:

def generate_balanced(n_total, seed=SEED):
    assert n_total % 10 == 0
    per = n_total // 10
    rng = np.random.default_rng(seed)
    labels = np.repeat(np.arange(10), per)
    noise = rng.normal(size=(n_total, LATENT)).astype(np.float32)
    perm = rng.permutation(n_total)
    labels, noise = labels[perm], noise[perm]
    imgs = generator.predict([noise, labels.reshape(-1, 1)], verbose=0)
    imgs = np.clip((imgs + 1) / 2, 0, 1).astype(np.float32)
    return imgs, labels.astype(np.int32)

X_syn, y_syn = generate_balanced(SYNTHETIC_TOTAL)
print('Synthetic shape', X_syn.shape, 'labels', np.bincount(y_syn).tolist())
assert len(X_syn) == SYNTHETIC_TOTAL
np.save(f'{OUT_DIR}/generated_samples/synthetic_images.npy', X_syn)
np.save(f'{OUT_DIR}/generated_samples/synthetic_labels.npy', y_syn)
pd.DataFrame({'id': np.arange(len(y_syn)), 'label': y_syn}).to_csv(
    f'{OUT_DIR}/generated_samples/synthetic_index.csv', index=False)

plt.figure(figsize=(10, 10))
plt.imshow(make_mosaic(X_syn, y_syn, per_class=10), cmap='gray')
plt.title('25k synthetic (or quick subset)'); plt.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/05_synthetic_grid.png', dpi=120); plt.show()

# Classifier agreement
pred_syn = clf_base.predict(X_syn, batch_size=512, verbose=0).argmax(1)
agree = (pred_syn == y_syn)
overall_agree = float(agree.mean())
per_class_agree = {int(c): float(agree[y_syn == c].mean()) for c in range(10)}
print(f'Overall classifier agreement: {overall_agree:.3f}')
print('Per-class:', {k: round(v, 3) for k, v in per_class_agree.items()})
print('NOTE: agreement is NOT proof of visual realism.')

# Simple near-duplicate rate on a subset
rng = np.random.default_rng(0)
n_sub = min(1500, len(X_syn))
idx = rng.choice(len(X_syn), n_sub, replace=False)
flat = X_syn[idx].reshape(n_sub, -1)
i1 = rng.integers(0, n_sub, 4000); i2 = rng.integers(0, n_sub, 4000)
mask = i1 != i2
dists = np.linalg.norm(flat[i1[mask]] - flat[i2[mask]], axis=1)
near_dup = float((dists < 1.0).mean())
print(f'Near-duplicate rate (L2 < 1): {near_dup:.4f}  mean L2={dists.mean():.2f}')

quality = dict(overall_agreement=overall_agree, per_class_agreement=per_class_agree,
               near_duplicate_rate=near_dup, mean_l2=float(dists.mean()), n=len(X_syn))
json.dump(quality, open(f'{OUT_DIR}/metrics/synthetic_quality.json', 'w'), indent=2)


## 7. Synthetic-ratio experiments (0 %, 10 %, 25 %)

Synthetic fraction = n_syn / (n_real + n_syn).  
Best ratio chosen by **validation accuracy only**.


In [ ]:

def mix(X_real, y_real, X_syn, y_syn, frac, seed=SEED):
    if frac <= 0:
        return X_real.copy(), y_real.copy()
    n_real = len(X_real)
    n_syn = int(round(frac / (1 - frac) * n_real))
    n_syn = min(n_syn, len(X_syn))
    rng = np.random.default_rng(seed)
    per = max(1, n_syn // 10)
    chosen = []
    for c in range(10):
        idx = np.where(y_syn == c)[0]
        take = min(per, len(idx))
        chosen.append(rng.choice(idx, size=take, replace=False))
    chosen = np.concatenate(chosen)
    if len(chosen) < n_syn:
        rest = np.setdiff1d(np.arange(len(X_syn)), chosen)
        extra = rng.choice(rest, size=n_syn - len(chosen), replace=False)
        chosen = np.concatenate([chosen, extra])
    Xc = np.concatenate([X_real, X_syn[chosen]])
    yc = np.concatenate([y_real, y_syn[chosen]])
    perm = rng.permutation(len(Xc))
    return Xc[perm], yc[perm]

ratio_rows = []
models = {}
for frac in SYNTHETIC_RATIOS:
    tag = f'ratio_{int(frac*100):02d}'
    print(f'\n=== {tag} (synthetic fraction = {frac:.0%}) ===')
    Xtr, ytr = mix(X_real, y_real, X_syn, y_syn, frac)
    print(f'  combined train size = {len(Xtr)}')
    m, hist, secs = train_classifier(Xtr, ytr, seed=0, name=tag)
    ev_v = evaluate(m, X_val, y_val)
    ev_t = evaluate(m, X_te, y_te)
    models[frac] = m
    weak_f1 = float(np.mean([ev_t['f1'][c] for c in WEAK]))
    ratio_rows.append(dict(
        ratio=frac, n_train=len(Xtr),
        val_acc=ev_v['acc'], val_macro_f1=ev_v['macro_f1'],
        test_acc=ev_t['acc'], test_macro_f1=ev_t['macro_f1'],
        weak3_f1=weak_f1, secs=secs))
    m.save(f'{OUT_DIR}/models/{tag}.keras')
    plot_cm(ev_t['cm'], f'{tag} CM (test)', f'{OUT_DIR}/figures/cm_{tag}.png')

ratio_df = pd.DataFrame(ratio_rows)
print(ratio_df.round(4).to_string(index=False))
ratio_df.to_csv(f'{OUT_DIR}/tables/ratio_comparison.csv', index=False)

# Select by VALIDATION accuracy
best_idx = int(ratio_df['val_acc'].idxmax())
BEST_RATIO = float(ratio_df.loc[best_idx, 'ratio'])
BEST_MODEL = models[BEST_RATIO]
print(f'\nBest ratio by validation accuracy: {BEST_RATIO:.0%}')


## 8. FGSM adversarial robustness audit

In [ ]:

def fgsm(model, x, y, eps):
    x = tf.convert_to_tensor(x, tf.float32)
    y = tf.convert_to_tensor(y, tf.int32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        loss = tf.reduce_sum(loss_per_sample(y, model(x, training=False)))
    g = tape.gradient(loss, x)
    return tf.clip_by_value(x + eps * tf.sign(g), 0.0, 1.0)

def attack_accuracy(model, X, y, eps, bs=256):
    correct = 0
    for i in range(0, len(X), bs):
        xb, yb = X[i:i+bs], y[i:i+bs]
        xa = fgsm(model, xb, yb, eps).numpy() if eps > 0 else xb
        pred = model.predict(xa, batch_size=bs, verbose=0).argmax(1)
        correct += int((pred == yb).sum())
    return correct / len(X)

# Sanity
xb, yb = X_te[:128], y_te[:128]
xa = fgsm(BEST_MODEL, xb, yb, 0.1).numpy()
assert np.abs(xa - xb).max() <= 0.1 + 1e-5
assert 0 <= xa.min() and xa.max() <= 1
print('FGSM sanity OK')

print('FGSM sweep on selected model:')
sweep = {}
for e in EPS_LIST:
    acc = attack_accuracy(BEST_MODEL, X_te, y_te, e)
    sweep[f'eps_{e:.2f}'] = acc
    print(f'  eps={e:.2f}  acc={acc:.4f}')
json.dump(sweep, open(f'{OUT_DIR}/metrics/fgsm_selected.json', 'w'), indent=2)

# Visual
n = 8
xb, yb = X_te[:n], y_te[:n]
xa = fgsm(BEST_MODEL, xb, yb, MAIN_EPS).numpy()
p0 = BEST_MODEL.predict(xb, verbose=0).argmax(1)
p1 = BEST_MODEL.predict(xa, verbose=0).argmax(1)
fig, axes = plt.subplots(3, n, figsize=(1.5*n, 5))
for i in range(n):
    axes[0, i].imshow(xb[i,...,0], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'y={yb[i]} p={p0[i]}', fontsize=7)
    axes[1, i].imshow(xa[i,...,0], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'adv p={p1[i]}', fontsize=7)
    axes[2, i].imshow((xa-xb)[i,...,0], cmap='bwr', vmin=-MAIN_EPS, vmax=MAIN_EPS)
    for r in range(3): axes[r, i].axis('off')
fig.suptitle(f'FGSM ε={MAIN_EPS} on selected model')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/06_fgsm_examples.png', dpi=120); plt.show()


## 9. Adversarial training

In [ ]:

def train_adv(X, y, seed, eps_train, epochs):
    model = build_cnn(seed, name='adv')
    opt = keras.optimizers.Adam(1e-3)
    ds = (tf.data.Dataset.from_tensor_slices((X, y.astype('int32')))
          .shuffle(len(X), seed=seed).batch(CFG['batch']))
    @tf.function
    def step(xb, yb):
        xadv = fgsm(model, xb, yb, eps_train)
        with tf.GradientTape() as t:
            loss = 0.5 * loss_obj(yb, model(xb, training=True)) + 0.5 * loss_obj(yb, model(xadv, training=True))
        opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
    best, best_w, wait = -1, None, 0
    for ep in range(epochs):
        for xb, yb in ds:
            step(xb, yb)
        vc = attack_accuracy(model, X_val, y_val, 0.0)
        va = attack_accuracy(model, X_val, y_val, eps_train)
        score = 0.5 * vc + 0.5 * va
        print(f'  adv epoch {ep+1}/{epochs}  val_clean={vc:.4f}  val_adv={va:.4f}')
        if score > best:
            best, best_w, wait = score, model.get_weights(), 0
        else:
            wait += 1
            if wait >= CFG['patience']:
                print('  early stop'); break
    model.set_weights(best_w)
    return model

Xadv_tr, yadv_tr = mix(X_real, y_real, X_syn, y_syn, BEST_RATIO)
print(f'Adversarial training on ratio {BEST_RATIO:.0%} data (n={len(Xadv_tr)})')
adv_model = train_adv(Xadv_tr, yadv_tr, seed=0, eps_train=MAIN_EPS, epochs=CFG['adv_epochs'])
adv_model.save(f'{OUT_DIR}/models/adversarial.keras')

print('FGSM sweep on adversarially trained model:')
adv_sweep = {}
for e in EPS_LIST:
    acc = attack_accuracy(adv_model, X_te, y_te, e)
    adv_sweep[f'eps_{e:.2f}'] = acc
    print(f'  eps={e:.2f}  acc={acc:.4f}')
json.dump(adv_sweep, open(f'{OUT_DIR}/metrics/fgsm_adv.json', 'w'), indent=2)


## 10. Final comparison tables

In [ ]:

final_rows = []
for name, model, ratio in [
    ('baseline_0pct', models[0.0], 0.0),
    (f'selected_{BEST_RATIO:.0%}', BEST_MODEL, BEST_RATIO),
    ('adversarial', adv_model, BEST_RATIO),
]:
    ev = evaluate(model, X_te, y_te)
    row = dict(model=name, synthetic_ratio=ratio,
               clean_test_acc=ev['acc'], macro_f1=ev['macro_f1'])
    for e in EPS_LIST:
        row[f'fgsm_{e:.2f}'] = attack_accuracy(model, X_te, y_te, e)
    final_rows.append(row)

final_df = pd.DataFrame(final_rows)
print(final_df.round(4).to_string(index=False))
final_df.to_csv(f'{OUT_DIR}/tables/final_comparison.csv', index=False)

# Accuracy vs epsilon plot
plt.figure(figsize=(7, 4.5))
for _, row in final_df.iterrows():
    ys = [row['clean_test_acc']] + [row[f'fgsm_{e:.2f}'] for e in EPS_LIST[1:]]
    plt.plot(EPS_LIST, ys, marker='o', label=row['model'])
plt.axhline(0.60, ls='--', c='gray', label='60 % target')
plt.axvline(0.10, ls=':', c='gray')
plt.xlabel('FGSM ε'); plt.ylabel('Test accuracy')
plt.title('Accuracy vs epsilon'); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/10_acc_vs_eps.png', dpi=120); plt.show()


## 11. Automatic target check

In [ ]:

best_clean = final_df['clean_test_acc'].max()
adv_row = final_df[final_df['model'] == 'adversarial'].iloc[0]
clean_drop_pp = 100 * (best_clean - adv_row['clean_test_acc'])
fgsm_ok = adv_row['fgsm_0.10'] >= 0.60
clean_ok = clean_drop_pp <= 2.0

print('=== TARGET CHECK (from YOUR run) ===')
print(f'Best clean test acc: {best_clean:.4f}')
print(f'Adversarial clean acc: {adv_row["clean_test_acc"]:.4f}  (drop {clean_drop_pp:+.2f} pp)  → {"PASS" if clean_ok else "MISS"}')
print(f'Adversarial FGSM@0.10: {adv_row["fgsm_0.10"]:.4f}  → {"PASS" if fgsm_ok else "MISS"}')
print(f'Weak-3 classes: {WEAK}')
print(f'Test set size unchanged: {len(X_te) == TEST_N}')
if QUICK_CHECK:
    print('*** QUICK_CHECK results – NOT FOR SUBMISSION ***')


## 12. Export

In [ ]:

# Save run config
json.dump(dict(CFG=CFG, SEED=SEED, WEAK=WEAK, BEST_RATIO=BEST_RATIO,
               QUICK_CHECK=QUICK_CHECK, TAG=TAG),
          open(f'{OUT_DIR}/run_config_{TAG}.json', 'w'), indent=2, default=str)

# Zip outputs
import shutil
shutil.make_archive(f'outputs_{TAG}', 'zip', OUT_DIR)
print('Outputs zip:', f'outputs_{TAG}.zip')
print('Figures:', sorted(os.listdir(f'{OUT_DIR}/figures')))
print('Tables:', sorted(os.listdir(f'{OUT_DIR}/tables')))
print('Done. Fill the report templates from the CSV/JSON files above.')
try:
    from google.colab import files
    files.download(f'outputs_{TAG}.zip')
except Exception:
    pass
